In [27]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [28]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [29]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


class ThresholdModel_rf:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


In [30]:
xgb = load_model("../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl")
lgbm = load_model("../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl")
rf = load_model("../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl")

✓ 모델 로드 완료: ../models\../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl
  모델 타입: ThresholdModel_rf


In [31]:
train, test = load_data()

X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=["ID"], axis=1)

# var3 처리
X_features["var3"] = X_features["var3"].replace(-999999, 2)

# train/val 분리
X_train, X_val, y_train, y_val = data_split(X_features, y_labels)


In [32]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# ======================================
# 1) Base 모델들의 예측값 생성
# ======================================

xgb_proba = xgb.predict_proba(X_val_scaled)[:, 1]
lgbm_proba = lgbm.predict_proba(X_val_scaled)[:, 1]
rf_proba = rf.predict_proba(X_val)[:, 1]  # RF는 원본 사용해도 무방

# 스택킹용 입력 데이터 생성
stack_val = np.vstack([xgb_proba, lgbm_proba, rf_proba]).T

In [ ]:
# ======================================
# 2) 메타 모델 정의 및 학습
# ======================================
meta = LogisticRegression(max_iter=2000)
meta.fit(stack_val, y_val)

# 메타 모델 예측
stack_proba = meta.predict_proba(stack_val)[:, 1]
stack_pred = (stack_proba > 0.5).astype(int)


===== 기본 Threshold(0.5) 스태킹 성능 =====
AUC : 0.8476702549734916
F1 : 0.24922311995027968


In [ ]:
# ======================================
# 3) 스택킹 모델 성능 평가
# ======================================
acc = accuracy_score(y_val, stack_pred)
f1 = f1_score(y_val, stack_pred)
auc = roc_auc_score(y_val, stack_proba)
cm = confusion_matrix(y_val, stack_pred)

print("\n===== Stacking Ensemble 성능 =====")
print(f"AUC : {auc:.4f}")
print(f"ACC : {acc:.4f}")
print(f"F1  : {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_val, stack_pred))



===== Best Threshold 적용 스태킹 성능 =====
AUC : 0.8476702549734916
F1 : 0.2449725776965265
Confusion Matrix:
 [[12324  2278]
 [  200   402]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.84      0.91     14602
           1       0.15      0.67      0.24       602

    accuracy                           0.84     15204
   macro avg       0.57      0.76      0.58     15204
weighted avg       0.95      0.84      0.88     15204



In [ ]:
# -----------------------------------------------
# 4) 메타 모델 기여도 출력
# -----------------------------------------------
print("\n[4] 메타 모델 기여도 (Coefficients)")
coef = meta.coef_[0]
for name, c in zip(["XGB", "LGBM", "RF"], coef):
    print(f"{name}: {c:.4f}")

print("\n===== 스태킹 앙상블 완료 =====")


[4] 메타 모델 기여도 (Coefficients)
XGB: 1.0046
LGBM: 6.9858
RF: 1.2859

===== 스태킹 앙상블 완료 =====
✓ 모델 저장 완료: ../models\StackingModel_XGB_LGBM_RF_20251124.pkl
  파일 크기: 0.00 MB

✓ 스태킹 메타모델 저장 완료!
